# Fig. S1, D to F and H: five-metric effect-size panel (incl. entropy)

Sub-panels under `Fig1/supp_3/<panel>/`. S3a multi-metric volcanos (Delta-psi, rho_W, NMI, MA);
S3b entropy_rho (concentrating vs spreading PAS usage); S3c composite-ranking components;
S3d directional x trajectory-pattern split. (rho_W / NMI / entropy_rho are reported as
effect-size descriptors only - no per-gene p-values cited.)


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

FIG = "Fig1/supp_3"
B = Path(f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_optimized")
df = pd.read_csv(B / "composite_ranking_optimized2_df5.csv")
df["nlp"] = -np.log10(df["padj"].clip(lower=1e-300))
cap = df.loc[np.isfinite(df.nlp), "nlp"].quantile(0.999)
df["nlp"] = df.nlp.clip(upper=cap * 1.05)
df["dir"] = np.where(
    (df.padj < 0.05) & (df.delta_psi < -0.1),
    "short",
    np.where((df.padj < 0.05) & (df.delta_psi > 0.1), "long", "NS"),
)
COL = {"short": S.COL_SHORT, "long": S.COL_LONG, "NS": S.COL_NS}
print(
    "trajectory_pattern:",
    df.loc[df.padj < 0.05, "trajectory_pattern"].value_counts().to_dict(),
)

In [ ]:
# S3a - multi-metric volcanos: Delta-psi, rho_W, NMI, and an MA (effect vs coverage)
import matplotlib.patches as mpatches

try:
    from adjustText import adjust_text

    HAS_AT = True
except ImportError:
    HAS_AT = False

ASD_HIGHLIGHT = {"Qk", "Dyrk1a", "Tcf4", "Prodh", "Nova2"}
# same top-10 selection as fig 1d (rho_W-gated significance, ranked by composite score)
_sig = df[(df.padj < 0.05) & (df.rho_W.abs() > 0.2)]
top_genes = _sig.nlargest(10, "composite_score_abs")

fig, axes = plt.subplots(2, 2, figsize=(S.FULL_W_PX / 300, 1300 / 300))


def vol(ax, xcol, xlab, vline=None):
    for cls in ["NS", "short", "long"]:
        m = (df["dir"] == cls).values
        ax.scatter(
            df.loc[m, xcol],
            df.loc[m, "nlp"],
            s=3,
            c=COL[cls],
            alpha=0.3 if cls == "NS" else 0.6,
            edgecolors="none",
            rasterized=True,
        )
    ax.axhline(-np.log10(0.05), color="gray", ls="--", lw=0.5)
    if vline is not None:
        for v in vline:
            ax.axvline(v, color="gray", ls=":", lw=0.5)
    ax.set_xlabel(xlab, fontsize=S.FONT_LABEL)
    ax.set_ylabel(r"$-\log_{10}$padj", fontsize=S.FONT_LABEL)
    ax.tick_params(labelsize=S.FONT_TICK)
    ax.spines[["top", "right"]].set_visible(False)


def annotate_top(ax, xcol, ycol="nlp"):
    """Overlay top-10 highlight genes (same set as fig 1d) and add italic labels."""
    texts = []
    for _, row in top_genes.iterrows():
        is_asd = row["gene_name"] in ASD_HIGHLIGHT
        xv, yv = row[xcol], row[ycol]
        col = "#8B0000" if is_asd else "#4d4d4d"
        col_scatter = "#8B0000" if is_asd else "#4d4d4d"
        ax.scatter(
            xv,
            yv,
            # s=22,
            # zorder=5,
            # c=col,
            # edgecolors="white",
            # linewidths=0.3,
            rasterized=True,
            s=3,
            marker="o",
            facecolors="none",
            edgecolors=col_scatter,
            linewidths=0.5,
            zorder=5,
        )
        texts.append(
            ax.text(
                xv,
                yv,
                row["gene_name"],
                fontsize=S.FONT_ANNOT - 1,
                color=col,
                fontweight="bold" if is_asd else "normal",
                fontstyle="italic",
                zorder=10,
            )
        )
    if HAS_AT and texts:
        adjust_text(
            texts,
            ax=ax,
            arrowprops=dict(arrowstyle="-", color="gray", lw=0.4),
            expand=(1.3, 1.4),
            force_text=(0.5, 0.8),
            # force_static=(0.2, 0.1),
            force_explode=(0.2, 0.6),
        )


vol(axes[0, 0], "delta_psi", r"$\Delta\psi$", vline=[-0.1, 0.1])
annotate_top(axes[0, 0], "delta_psi")

vol(axes[0, 1], "rho_W", r"$\rho$ (length vs pseudotime)", vline=[-0.2, 0.2])
annotate_top(axes[0, 1], "rho_W")

vol(axes[1, 0], "nmi", "NMI (non-monotonicity)")
annotate_top(axes[1, 0], "nmi")

# MA: effect vs coverage (Delta-psi should not be coverage-driven)
axm = axes[1, 1]
lu = np.log10(df["total_umis"].clip(lower=1))
for cls in ["NS", "short", "long"]:
    m = (df["dir"] == cls).values
    axm.scatter(
        lu[m],
        df.loc[m, "delta_psi"],
        s=3,
        c=COL[cls],
        alpha=0.3 if cls == "NS" else 0.6,
        edgecolors="none",
        rasterized=True,
    )
axm.axhline(0, color="gray", ls="--", lw=0.5)
axm.set_xlabel(r"$\log_{10}$ total UMIs", fontsize=S.FONT_LABEL)
axm.set_ylabel(r"$\Delta\psi$", fontsize=S.FONT_LABEL)
axm.tick_params(labelsize=S.FONT_TICK)
axm.spines[["top", "right"]].set_visible(False)
axm.set_title("MA: effect vs coverage", fontsize=S.FONT_ANNOT, pad=2)

# annotate top genes on MA (x = log10 total_umis, y = delta_psi)
ma_texts = []
for _, row in top_genes.iterrows():
    is_asd = row["gene_name"] in ASD_HIGHLIGHT
    xv = np.log10(max(row["total_umis"], 1))
    yv = row["delta_psi"]
    col = "#8B0000" if is_asd else "#4d4d4d"
    col_scatter = "#8B0000" if is_asd else "#4d4d4d"
    axm.scatter(
        xv,
        yv,
        s=3,
        marker="o",
        facecolors="none",
        edgecolors=col_scatter,
        linewidths=0.5,
        zorder=6,
    )
    ma_texts.append(
        axm.text(
            xv,
            yv,
            row["gene_name"],
            fontsize=S.FONT_ANNOT - 1,
            color=col,
            fontweight="bold" if is_asd else "normal",
            fontstyle="italic",
            zorder=10,
        )
    )
if HAS_AT and ma_texts:
    adjust_text(
        ma_texts,
        ax=axm,
        arrowprops=dict(arrowstyle="-", color="gray", lw=0.4),
        expand=(1.3, 1.4),
        force_text=(1.5, 1.8),
        # force_static=(1.4, 1.4),
        force_explode=(0.2, 0.6),
    )

legend_handles = [
    mpatches.Patch(color=COL["short"], label="shortening (padj<0.05, $\\Delta\\psi$<-0.1)"),
    mpatches.Patch(color=COL["long"], label="lengthening (padj<0.05, $\\Delta\\psi$>0.1)"),
    mpatches.Patch(color=COL["NS"], label="NS"),
]
fig.legend(
    handles=legend_handles,
    fontsize=S.FONT_LEGEND,
    frameon=False,
    loc="lower center",
    ncol=3,
    bbox_to_anchor=(0.5, -0.01),
)
fig.subplots_adjust(left=0.10, right=0.97, top=0.96, bottom=0.14, hspace=0.32, wspace=0.30)
S.save_and_show(fig, "fig_s3a_multimetric_volcanos", figure=FIG)

In [ ]:
# S3b - entropy_rho: PAS usage concentrating (rho<0) vs spreading (rho>0)
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 720 / 300))
e = df["entropy_rho"].dropna()
neg = int((e < 0).sum())
pos = int((e > 0).sum())
# Symmetric range ensures 0 is exactly a bin edge (midpoint of linspace with odd count).
# Without this, the bin straddling 0 gets its counts split → artificial drop near 0.
limit = max(abs(e.min()), abs(e.max()))
bins = np.linspace(-limit, limit, 81)  # 80 bins, uniform width, 0 is exact bin edge
ax.hist(
    e[e < 0],
    bins=bins,
    color="#3690c0",
    alpha=0.8,
    edgecolor="none",
    label=f"concentrating ($\\rho$<0): {neg:,}",
)
ax.hist(
    e[e > 0],
    bins=bins,
    color="#ef6548",
    alpha=0.8,
    edgecolor="none",
    label=f"spreading ($\\rho$>0): {pos:,}",
)
ax.axvline(0, color="k", lw=0.6)
ax.set_xlabel(r"entropy $\rho$  (PAS-usage spread vs pseudotime)", fontsize=S.FONT_LABEL)
ax.set_ylabel("genes", fontsize=S.FONT_LABEL)
ax.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="upper left")
ax.set_title("entropy_rho - orthogonal to the centroid shift", fontsize=S.FONT_ANNOT, pad=3)
ax.tick_params(labelsize=S.FONT_TICK)
ax.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.12, right=0.96, top=0.91, bottom=0.15)
S.save_and_show(fig, "fig_s3b_entropy_rho", figure=FIG)

In [ ]:
# S3c - composite-ranking components (50% scaled-padj / 25% rank-Delta-psi / 25% rank-rho_W)
fig, axes = plt.subplots(1, 3, figsize=(S.FULL_W_PX / 300, 620 / 300))
pairs = [
    (
        "scaled_neglog10_padj",
        "rank_delta_psi",
        "scaled -log10 padj",
        "rank |Delta-psi|",
    ),
    ("scaled_neglog10_padj", "rank_abs_rho_W", "scaled -log10 padj", "rank |rho_W|"),
    ("rank_delta_psi", "rank_abs_rho_W", "rank |Delta-psi|", "rank |rho_W|"),
]
for ax, (xc, yc, xl, yl) in zip(axes, pairs):
    sc = ax.scatter(
        df[xc],
        df[yc],
        s=3,
        c=df["composite_score_abs"],
        cmap="viridis",
        alpha=0.6,
        edgecolors="none",
        rasterized=True,
    )
    ax.set_xlabel(xl, fontsize=S.FONT_TICK)
    ax.set_ylabel(yl, fontsize=S.FONT_TICK)
    ax.tick_params(labelsize=S.FONT_TICK - 1)
    ax.spines[["top", "right"]].set_visible(False)
cb = fig.colorbar(sc, ax=axes, fraction=0.025, pad=0.02)
cb.set_label("composite score", fontsize=S.FONT_TICK)
cb.ax.tick_params(labelsize=S.FONT_TICK - 1)
fig.suptitle(
    "Composite = 50% scaled -log10 padj  +  25% rank|Delta-psi|  +  25% rank|rho_W|  (complementary axes)",
    fontsize=S.FONT_ANNOT,
    y=0.99,
)
fig.subplots_adjust(left=0.06, right=0.88, top=0.86, bottom=0.18, wspace=0.32)
S.save_and_show(fig, "fig_s3c_composite_components", figure=FIG)

In [ ]:
# S3d - directional x trajectory-pattern split (among significant genes)
sig = df[df.padj < 0.05].copy()
sig["pat"] = np.where(
    sig.trajectory_pattern.astype(str).str.startswith("Complex"), "complex", "monotonic"
)
sig["dir2"] = np.where(
    sig.delta_psi < 0,
    "shortening",
    np.where(sig.delta_psi > 0, "lengthening", "stable"),
)
ct = (
    pd.crosstab(sig["dir2"], sig["pat"])
    .reindex(index=["shortening", "lengthening"], columns=["monotonic", "complex"])
    .fillna(0)
)
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 720 / 300))
y = np.arange(2)
ax.barh(y, ct["monotonic"], color="#9970ab", label="monotonic")
ax.barh(
    y,
    ct["complex"],
    left=ct["monotonic"],
    color="#5aae61",
    label="complex (non-monotonic)",
)
for i, idx in enumerate(ct.index):
    ax.text(
        ct.loc[idx].sum() + 8,
        y[i],
        f"{int(ct.loc[idx, 'monotonic'])} mono / {int(ct.loc[idx, 'complex'])} cplx",
        va="center",
        ha="left",
        fontsize=S.FONT_ANNOT - 1,
    )
ax.set_yticks(y)
ax.set_yticklabels(ct.index)
ax.set_xlabel("significant genes", fontsize=S.FONT_LABEL)
n_complex = int((sig.pat == "complex").sum())
ax.set_title(
    f"{100 * n_complex / len(sig):.1f}% of significant genes ({n_complex:,}/{len(sig):,}) "
    f"are complex (non-monotonic)",
    fontsize=S.FONT_ANNOT - 0.5,
    pad=3,
)
ax.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=S.FONT_TICK)
ax.set_xlim(0, ct.values.sum(axis=1).max() * 1.22)
fig.subplots_adjust(left=0.20, right=0.96, top=0.90, bottom=0.15)
S.save_and_show(fig, "fig_s3d_direction_x_pattern", figure=FIG)

In [ ]:
# S3e - APA regulatory-mode map: centroid shift (Delta-psi) x concentration (entropy_rho)
sig = df[df.padj < 0.05].copy()
sig["q"] = np.where(
    sig.delta_psi >= 0,
    np.where(sig.entropy_rho < 0, "commit-long", "diversify-long"),
    np.where(sig.entropy_rho < 0, "commit-short", "diversify-short"),
)
QC = {
    "commit-long": "#b2182b",
    "diversify-long": "#f4a582",
    "commit-short": "#2166ac",
    "diversify-short": "#92c5de",
}
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 760 / 300))
for q, c in QC.items():
    m = (sig.q == q).values
    ax.scatter(
        sig.loc[m, "delta_psi"],
        sig.loc[m, "entropy_rho"],
        s=4,
        c=c,
        alpha=0.5,
        edgecolors="none",
        rasterized=True,
        label=f"{q} ({int(m.sum())})",
    )
ax.axhline(0, c="k", lw=0.6)
ax.axvline(0, c="k", lw=0.6)
for g in ["Qk", "Prodh", "Clptm1", "Ppp1r37", "Tcf4"]:
    r = df[df.gene_name == g]
    if len(r):
        ax.scatter(
            r.delta_psi,
            r.entropy_rho,
            s=20,
            facecolors="none",
            edgecolors="k",
            linewidths=0.7,
            zorder=5,
        )
        ax.annotate(
            f"$\\it{{{g}}}$",
            (r.delta_psi.iloc[0], r.entropy_rho.iloc[0]),
            fontsize=S.FONT_ANNOT - 1,
            ha="left",
            va="bottom",
        )
ax.set_xlabel(r"$\Delta\psi$  (centroid:  + lengthen / - shorten)", fontsize=S.FONT_LABEL)
ax.set_ylabel(r"entropy $\rho$  ($-$ commit / $+$ diversify)", fontsize=S.FONT_LABEL)
ax.set_title(
    "APA regulatory-mode map\n(two orthogonal axes: length shift x isoform commitment)",
    fontsize=S.FONT_ANNOT,
    pad=3,
)
ax.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="upper center", ncol=2, columnspacing=0.8)
ax.tick_params(labelsize=S.FONT_TICK)
ax.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.15, right=0.96, top=0.86, bottom=0.14)
S.save_and_show(fig, "fig_s3e_mode_map", figure=FIG)

In [ ]:
# S3e (significance-weighted) - same Delta-psi x entropy_rho map, dot opacity proportional to
# significance (-log10 padj, capped) so the confident hits show through; tests whether the most
# significant genes carry an entropy bias.
import matplotlib.colors as mcolors
from scipy.stats import spearmanr

d2 = df.dropna(subset=["delta_psi", "entropy_rho"]).copy()
nl = -np.log10(d2["padj"].clip(lower=1e-300))
_sig = d2["padj"] < 0.05
CAP = float(np.nanpercentile(nl[_sig], 75))  # cap so very-significant genes don't wash out the rest
alpha = 0.06 + 0.94 * np.clip(nl / CAP, 0, 1)  # floor 0.06 (non-sig fade); padj=0.05 -> ~0.31
d2["q"] = np.where(
    d2.delta_psi >= 0,
    np.where(d2.entropy_rho < 0, "commit-long", "diversify-long"),
    np.where(d2.entropy_rho < 0, "commit-short", "diversify-short"),
)
QC = {
    "commit-long": "#b2182b",
    "diversify-long": "#f4a582",
    "commit-short": "#2166ac",
    "diversify-short": "#92c5de",
}
rgba = np.array([mcolors.to_rgba(QC[q]) for q in d2["q"]])
rgba[:, 3] = alpha
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 820 / 300))
o = np.argsort(alpha)  # draw faint first, significant on top
ax.scatter(
    d2["delta_psi"].values[o],
    d2["entropy_rho"].values[o],
    s=5,
    c=rgba[o],
    edgecolors="none",
    rasterized=True,
)
ax.axhline(0, c="k", lw=0.6)
ax.axvline(0, c="k", lw=0.6)
for g in ["Qk", "Prodh", "Clptm1", "Ppp1r37", "Tcf4"]:
    r = df[df.gene_name == g]
    if len(r):
        ax.scatter(
            r.delta_psi,
            r.entropy_rho,
            s=20,
            facecolors="none",
            edgecolors="k",
            linewidths=0.7,
            zorder=5,
        )
        ax.annotate(
            f"$\\it{{{g}}}$",
            (r.delta_psi.iloc[0], r.entropy_rho.iloc[0]),
            fontsize=S.FONT_ANNOT - 1,
            ha="left",
            va="bottom",
        )
rho, pp = spearmanr(nl[_sig], d2.loc[_sig, "entropy_rho"])
ax.text(
    0.02,
    0.02,
    f"sig genes: Spearman($-\\log_{{10}}$padj, entropy $\\rho$) = {rho:+.2f} (p={pp:.2f})\n-> no entropy bias with significance",
    transform=ax.transAxes,
    fontsize=S.FONT_ANNOT - 1.5,
    va="bottom",
    ha="left",
)
from matplotlib.lines import Line2D

leg = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=QC[k], markersize=4, label=k)
    for k in QC
]
ax.legend(
    handles=leg,
    fontsize=S.FONT_LEGEND,
    frameon=False,
    loc="upper center",
    ncol=2,
    columnspacing=0.8,
)
ax.set_xlabel(r"$\Delta\psi$  (centroid:  + lengthen / - shorten)", fontsize=S.FONT_LABEL)
ax.set_ylabel(r"entropy $\rho$  ($-$ commit / $+$ diversify)", fontsize=S.FONT_LABEL)
ax.set_title(
    f"mode map, opacity $\\propto$ significance ($-\\log_{{10}}$ padj, cap {CAP:.0f}); opaque = more significant",
    fontsize=S.FONT_ANNOT - 0.5,
    pad=3,
)
ax.tick_params(labelsize=S.FONT_TICK)
ax.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.15, right=0.96, top=0.88, bottom=0.13)
S.save_and_show(fig, "fig_s3e_mode_map_pweighted", figure=FIG)

In [ ]:
# S3f - isoform-commitment vignette: a strong concentrator (lengthen + concentrate)
import scanpy as sc
from scipy.stats import rankdata

SMOOTH = f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/UTR_count_table_genelevel_harmony_proportions_lineage_smoothed.h5ad"
ad = sc.read_h5ad(SMOOTH)
pt = ad.obs["dpt_pseudotime"].values
GENE = "Ppp1r37"
m = (ad.var["gene_name"] == GENE).values
sub = ad[:, m]
pr = sub.layers["smoothed_proportions"]
pr = pr.toarray() if hasattr(pr, "toarray") else np.asarray(pr)
sizes = sub.var["utr_size"].values.astype(float)
o = np.argsort(sizes)
pr = pr[:, o]
sizes = sizes[o]
K = pr.shape[1]
pp = np.clip(pr, 1e-12, 1.0)
Hcell = -(pp * np.log(pp)).sum(1) / np.log(K)  # normalised entropy in [0,1]
valid = ~np.isnan(pt)
b = pd.cut(pt[valid], bins=40, labels=False)
cen = np.array([pt[valid][b == i].mean() for i in range(40)])
mp = np.array([pr[valid][b == i].mean(axis=0) for i in range(40)])
He = np.array([Hcell[valid][b == i].mean() for i in range(40)])
keep = ~np.isnan(cen)
cen, mp, He = cen[keep], mp[keep], He[keep]
mp = mp / mp.sum(1, keepdims=True)
fig, ax = plt.subplots(
    1, 2, figsize=(S.FULL_W_PX / 300, 540 / 300), gridspec_kw={"width_ratios": [1.1, 1]}
)
cols = list(__import__("seaborn").color_palette("RdYlBu_r", K))
ax[0].stackplot(cen, mp.T, colors=cols, alpha=0.88)
ax[0].set_xlim(cen.min(), cen.max())
ax[0].set_ylim(0, 1)
ax[0].set_xlabel("pseudotime", fontsize=S.FONT_LABEL)
ax[0].set_ylabel("UTR-site proportion", fontsize=S.FONT_LABEL)
ax[0].set_title(
    f"$\\it{{{GENE}}}$ (K={K}): usage concentrates onto the long isoform",
    fontsize=S.FONT_ANNOT,
)
ax[1].plot(cen, He, "-", color="#6a51a3", lw=1.6)
ax[1].set_xlabel("pseudotime", fontsize=S.FONT_LABEL)
ax[1].set_ylabel("normalised PAS-usage entropy", fontsize=S.FONT_LABEL)
ax[1].set_title("entropy decreases = isoform commitment", fontsize=S.FONT_ANNOT)
for a in ax:
    a.tick_params(labelsize=S.FONT_TICK)
    a.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.08, right=0.97, top=0.86, bottom=0.17, wspace=0.32)
S.save_and_show(fig, "fig_s3f_commitment_vignette", figure=FIG)
del ad

### Half-width x quarter-height version of the commitment vignette

`fig_s3f_commitment_vignette__half_quarter` (HALF_W x QUARTER_H). Reuses the binned data
(cen/mp/He/K/cols/GENE) computed above; only the size, fonts and titles are compacted.


In [ ]:
# --- Half-width x quarter-height version of the commitment vignette ---
# Reuses cen, mp, He, K, cols, GENE computed above (no h5ad reload).
fig, ax = plt.subplots(
    1,
    2,
    figsize=(S.HALF_W_IN, S.QUARTER_H_IN),
    gridspec_kw={"width_ratios": [1.1, 1], "wspace": 0.42},
)
ax[0].stackplot(cen, mp.T, colors=cols, alpha=0.88)
ax[0].set_xlim(cen.min(), cen.max())
ax[0].set_ylim(0, 1)
ax[0].set_xlabel("pseudotime", fontsize=S.FONT_TICK)
ax[0].set_ylabel("UTR-site proportion", fontsize=S.FONT_TICK)
ax[0].set_title(f"$\\it{{{GENE}}}$ (K={K}): commits to long isoform", fontsize=S.FONT_ANNOT)
ax[1].plot(cen, He, "-", color="#6a51a3", lw=1.4)
ax[1].set_xlabel("pseudotime", fontsize=S.FONT_TICK)
ax[1].set_ylabel("norm. PAS entropy", fontsize=S.FONT_TICK)
ax[1].set_title("entropy down = commitment", fontsize=S.FONT_ANNOT)
for a in ax:
    a.tick_params(labelsize=S.FONT_TICK - 1)
    a.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.11, right=0.97, top=0.85, bottom=0.20, wspace=0.42)
S.save_and_show(fig, "fig_s3f_commitment_vignette__half_quarter", figure=FIG)

### Half-width x quarter-height commitment vignette on the Fig 1e uniform x-axis (equal-cell + cell-type boxes)

Same panel as `..._half_quarter`, but the x-axis matches the standard Fig 1e uniform exemplar:
equal-cell (ECDF) pseudotime, with a cell-type-majority colour bar on the stacked-area subplot
(no pseudotime ticks) and the entropy subplot on the same equal-cell (quantile) axis. Saved as
`fig_s3f_commitment_vignette__half_quarter_ctbar`.


In [ ]:
# --- S3f half-width x quarter-height on the Fig 1e uniform x-axis (equal-cell + cell-type boxes) ---
# The entropy subpanel also gets a cell-type box strip below it (like the stacked area).
import scanpy as sc
import seaborn as _sns
from matplotlib.lines import Line2D as _L2D
from scipy.stats import rankdata

_SM = f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/UTR_count_table_genelevel_harmony_proportions_lineage_smoothed.h5ad"
_ad = sc.read_h5ad(_SM)
_pt = _ad.obs["dpt_pseudotime"].values.astype(float)
_ct = _ad.obs["celltype"].astype(str).values
GENE = "Ppp1r37"
_m = (_ad.var["gene_name"] == GENE).values
_sub = _ad[:, _m]
_pr = _sub.layers["smoothed_proportions"]
_pr = _pr.toarray() if hasattr(_pr, "toarray") else np.asarray(_pr)
_sz = _sub.var["utr_size"].values.astype(float)
_o = np.argsort(_sz)
_pr = _pr[:, _o]
K = _pr.shape[1]
_pp = np.clip(_pr, 1e-12, 1.0)
_H = -(_pp * np.log(_pp)).sum(1) / np.log(K)
_v = ~np.isnan(_pt)
_u = np.full_like(_pt, np.nan)
_u[_v] = (rankdata(_pt[_v], method="average") - 0.5) / _v.sum()
_CT = S.MOUSE_LINEAGE_ORDER
_uv, _ctv, _prv, _Hv = _u[_v], _ct[_v], _pr[_v], _H[_v]
_b = pd.cut(_uv, bins=40, labels=False)
cen = np.array([_uv[_b == i].mean() for i in range(40)])
mp = np.array([_prv[_b == i].mean(0) for i in range(40)])
He = np.array([_Hv[_b == i].mean() for i in range(40)])
_keep = ~np.isnan(cen)
cen, mp, He = cen[_keep], mp[_keep], He[_keep]
mp = mp / mp.sum(1, keepdims=True)
cols = list(_sns.color_palette("RdYlBu_r", K))


def _ct_ranges_u():
    out = []
    for c in _CT:
        cu = _uv[_ctv == c]
        if len(cu):
            out.append((c, float(cu.min()), float(cu.max())))
    return out


fig, ax = plt.subplots(
    1,
    2,
    figsize=(S.HALF_W_IN, S.QUARTER_H_IN),
    gridspec_kw={"width_ratios": [1.1, 1], "wspace": 0.42},
)
# left: stacked area + cell-type boxes
ax[0].stackplot(cen, mp.T, colors=cols, alpha=0.88)
ax[0].set_xlim(0, 1)
ax[0].set_ylim(-0.14, 1.0)
for c, a, b in _ct_ranges_u():
    ax[0].fill_between(
        [a, b], -0.13, -0.02, color=S.paper_palette_celltypes[c], alpha=0.9, linewidth=0
    )
ax[0].axhline(0, color="grey", lw=0.4, alpha=0.3)
ax[0].set_ylabel("UTR-site proportion", fontsize=S.FONT_TICK)
ax[0].set_title(f"$\\it{{{GENE}}}$ (K={K}): commits to long isoform", fontsize=S.FONT_ANNOT)
ax[0].set_xticks([])
ax[0].tick_params(axis="x", bottom=False, labelbottom=False)
for sp in ("top", "right", "bottom"):
    ax[0].spines[sp].set_visible(False)
# right: entropy + cell-type boxes (strip below, extended ylim)
ax[1].plot(cen, He, "-", color="#6a51a3", lw=1.4)
ax[1].set_xlim(0, 1)
lo = float(He.min())
hi = float(He.max())
span = (hi - lo) or 1.0
for c, a, b in _ct_ranges_u():
    ax[1].fill_between(
        [a, b],
        lo - 0.13 * span,
        lo - 0.02 * span,
        color=S.paper_palette_celltypes[c],
        alpha=0.9,
        linewidth=0,
    )
ax[1].set_ylim(lo - 0.14 * span, hi)
ax[1].axhline(lo, color="grey", lw=0.4, alpha=0.3)
ax[1].set_ylabel("norm. PAS entropy", fontsize=S.FONT_TICK)
ax[1].set_title("entropy down = commitment", fontsize=S.FONT_ANNOT)
ax[1].set_xticks([])
ax[1].tick_params(axis="x", bottom=False, labelbottom=False)
for sp in ("top", "right", "bottom"):
    ax[1].spines[sp].set_visible(False)
ax[1].tick_params(labelsize=S.FONT_TICK - 1)
present = [c for c, *_ in _ct_ranges_u()]
ct_h = [
    _L2D(
        [0],
        [0],
        marker="s",
        color="w",
        markerfacecolor=S.paper_palette_celltypes[c],
        markersize=4,
        label=c,
    )
    for c in present
]
fig.legend(
    handles=ct_h,
    loc="lower center",
    ncol=len(present),
    fontsize=S.FONT_LEGEND - 1.5,
    frameon=False,
    bbox_to_anchor=(0.5, -0.01),
)
fig.subplots_adjust(left=0.11, right=0.97, top=0.85, bottom=0.20, wspace=0.42)
S.save_and_show(fig, "fig_s3f_commitment_vignette__half_quarter_ctbar", figure=FIG)
del _ad